# 모든 토큰이 참조하는 Shared Expert와 특정 지식만 담당하는 Routed Expert 레이어 분리 코딩

> 본 노트북은 SYLLABUS.md에 기반하여 자동 생성된 뼈대(Skeleton) 파일입니다. 상세한 이론, 수식 및 코드는 추가로 구현되어야 합니다.

## 1. 개요 및 학습 목표
이 노트북에서는 해당 주제에 대한 핵심 개념을 다룹니다.

## 2. 핵심 이론 및 수학적 원리
- 수식 및 상세한 동작 원리를 여기에 기록합니다.

## 3. 실습 코드 구현
아래 셀을 통해 파이썬 및 관련 프레임워크 코드를 직접 작성하고 실행해 볼 수 있습니다.

In [ ]:
# 실습을 위한 기본 라이브러리 임포트
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")


---
## Q1: Shared Expert + Routed Expert 구조 정의

Shared Expert MoE 구조를 정의하세요. Shared Expert는 모든 토큰에 항상 적용되고, Routed Expert는 Router에 의해 선택적으로 적용됩니다.

**요구사항:**
- `d_model=256`, `d_ff=512`, `num_routed=4` 설정
- `shared_expert`: `Dense(d_ff, relu) → Dense(d_model)` 레이어
- `routed_experts`: `num_routed`개의 동일 구조 Expert 리스트
- `router`: `Dense(num_routed)` 레이어
- 각 컴포넌트의 파라미터 수를 출력하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import tensorflow as tf

d_model, d_ff, num_routed = 256, 512, 4

shared_expert = tf.keras.Sequential([
    tf.keras.layers.Dense(d_ff, activation='relu'),
    tf.keras.layers.Dense(d_model)
], name='shared_expert')

routed_experts = [
    tf.keras.Sequential([
        tf.keras.layers.Dense(d_ff, activation='relu'),
        tf.keras.layers.Dense(d_model)
    ]) for _ in range(num_routed)
]
router = tf.keras.layers.Dense(num_routed)

# Build
x_dummy = tf.random.normal([1, 1, d_model])
_ = shared_expert(x_dummy)
for e in routed_experts: _ = e(x_dummy)
_ = router(x_dummy)

print(f"Shared Expert params: {shared_expert.count_params():,}")
print(f"Routed Experts params: {sum(e.count_params() for e in routed_experts):,}")
print(f"Router params: {router.count_params():,}")
```
</details>

---
## Q2: Top-2 라우팅 및 Shared+Routed 합산

전체 Shared Expert MoE 순전파를 구현하세요.

**요구사항:**
- `shared_out = shared_expert(x)`
- `scores = softmax(router(x))` → `top_k(k=2)` 로 Top-2 선택 및 재정규화
- Top-2 Expert 출력을 가중합하여 `routed_out` 계산
- `final_out = shared_out + routed_out`
- `x = tf.random.normal([2, 10, 256])`으로 테스트하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import tensorflow as tf

x = tf.random.normal([2, 10, d_model])
top_k = 2

shared_out = shared_expert(x)
scores = tf.nn.softmax(router(x), axis=-1)
top_vals, top_idx = tf.math.top_k(scores, k=top_k)
top_weights = top_vals / tf.reduce_sum(top_vals, axis=-1, keepdims=True)

all_expert_out = tf.stack([e(x) for e in routed_experts], axis=2)  # (B,T,E,D)
one_hot = tf.one_hot(top_idx, num_routed)  # (B,T,K,E)
routed_out = tf.zeros_like(x)
for k in range(top_k):
    mask = one_hot[:, :, k, :]
    sel = tf.reduce_sum(all_expert_out * mask[:,:,:,tf.newaxis], axis=2)
    routed_out = routed_out + top_weights[:, :, k:k+1] * sel

final_out = shared_out + routed_out
print(f"출력 shape: {final_out.shape}")
```
</details>

---
## Q3: Expert 선택 분포 시각화

Router가 각 Expert를 몇 번이나 선택했는지 분포를 시각화하고, 균등 분배 기준과 비교하세요.

**요구사항:**
- `top_idx` (B,T,K) 텐서에서 전체 선택 횟수를 Expert별로 집계하세요.
- 막대 그래프로 시각화하고, 균등 분배 기준선(`B*T*K / num_routed`)을 빨간 점선으로 표시하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import matplotlib.pyplot as plt
import numpy as np

counts = np.zeros(num_routed)
for idx in top_idx.numpy().reshape(-1):
    counts[idx] += 1

expected = len(top_idx.numpy().reshape(-1)) / num_routed
plt.figure(figsize=(6, 3))
plt.bar([f'E{i}' for i in range(num_routed)], counts, color='steelblue')
plt.axhline(expected, color='red', linestyle='--', label=f'균등 기준 ({expected:.0f})')
plt.title('Routed Expert 선택 분포')
plt.ylabel('선택 횟수'); plt.legend()
plt.tight_layout(); plt.show()
```
</details>